# Session 1 — Naive Decoding

No KV cache: every new token forces a full forward pass over the whole sequence so far. This is the baseline every later session improves on.

In [1]:
import sys; sys.path.insert(0, '../src')
import torch
from torch import nn
from kv_cache_variants.naive_decode import TinyCausalAttention, generate_naive

torch.manual_seed(0)
vocab, d_model = 50, 32
embed = nn.Embedding(vocab, d_model)
lm_head = nn.Linear(d_model, vocab, bias=False)
attn = TinyCausalAttention(d_model)

prompt = torch.randint(0, vocab, (1, 4))
out = generate_naive(attn, prompt, embed, lm_head, max_new_tokens=8)
print('generated ids:', out)

generated ids: tensor([[ 7,  9, 18, 28, 22, 15, 14,  3,  7,  7,  7,  7]])


In [2]:
# Cost intuition: step t recomputes attention over t tokens -> O(n^2) total work for n steps.
# Session 2 introduces the memory math for caching K/V instead of recomputing them.